### Update model and calculate metrics for Biolog experiments

#### Import libraries
----

In [1]:
import json
import os
from copy import deepcopy
cwd = os.getcwd().split("\\")[:-1]
cwd = "\\".join(cwd)
os.chdir(cwd)
print(cwd)

import cobra
import polars as pl
from scripts.utils import update_metabolites, update_reactions, write_excel
from sklearn.metrics import confusion_matrix, matthews_corrcoef
from xlsxwriter import Workbook

C:\Users\Local_Admin\projects\h_lacustris


ModuleNotFoundError: No module named 'scripts.utils'

#### Load Biolog data
----

In [ ]:
biolog_dict_file = "data/external/biolog_to_bigg_dict.xlsx" # Biolog to bigg dict
biolog_data_file = "data/2_processed/supplementary_tables.xlsx" # Experiment Data

biolog_dict = pl.read_excel(biolog_dict_file) # Biolog to bigg dict
biolog_data = pl.read_excel(biolog_data_file, sheet_name="decision") # Experiment Data
biolog_data = biolog_data.filter(pl.col("method") == "all") # Get growth decision based on the three methods
# Add bigg identifiers to the data
biolog_data = biolog_data.join(
    biolog_dict.select("plate", "metabolite", "peptide", "new_bigg", "bigg", "metacyc"),
    on=["plate", "metabolite"]
)
biolog_data

#### Load the model
----

In [10]:
model_file = "models/draft/v0.0.1/hlacustris.xml"
model_excel = "models/draft/v0.0.1/hlacustris.xlsx"

model = cobra.io.read_sbml_model(model_file)
metabolites = pl.read_excel(model_excel, sheet_name="metabolites")
model

Set parameter Username
Set parameter LicenseID to value 2744854
Academic license - for non-commercial use only - expires 2026-11-25


Name,iHlct
Memory address,1e62cf13b60
Number of metabolites,1766
Number of reactions,2294
Number of genes,762
Number of groups,0
Objective expression,1.0*BIOMASS_hlacus_auto - 1.0*BIOMASS_hlacus_auto_reverse_375bd
Compartments,"mitochondrion, chloroplast, cytoplasm, glyoxysome, extracellular, thylakoid"


#### Find biolog mets ready for simulation
---

In [46]:
# Find which biolog metabolites are in any compartments
# dict of mets by compartment
mets_in_comps = dict(zip(list(model.compartments), [[], [], [], [], [], []]))
# If we find the met_id in the model we add it to the biolog_data
# Otherwise we add None
for row in biolog_data.iter_rows(named=True):
    bigg_id = row["bigg"]
    if bigg_id:
        for comp in model.compartments:
            # Build the met id for each compartment eg. "met__L" + "_x"
            met_id = bigg_id + "_" + comp
            if met_id in metabolites["id"]:
                mets_in_comps[comp].append(met_id)
            else:
                mets_in_comps[comp].append(None)
    else:
        for comp in model.compartments:
            mets_in_comps[comp].append(None)
# Append the columns to the biolog_data
biolog_data = biolog_data.with_columns(
    [pl.Series(name=key, values=value) for key, value in mets_in_comps.items()]
)

# Split the metabolites by:
#   1. Metabolites not in the model
#   2. Metabolites in the model but not in the extracellular compartment
#   3. Metabolites ready for simulations
not_in_model_mask = [all(not bool(row[c]) for c in model.compartments)
                     for row in biolog_data.iter_rows(named=True)]
not_in_e_mask = [any(bool(row[c]) for c in model.compartments) and not bool(row["e"])
                 for row in biolog_data.iter_rows(named=True)]
# These mets are ready to run simulations
in_e_mask = [bool(row["e"]) for row in biolog_data.iter_rows(named=True)] 
print(f"{sum(in_e_mask)} metabolites ready for simulations")

56 metabolites ready for simulations


In [47]:
# Save the metabolites
file = "data/1_interim/biolog_mets_in_model.xlsx"
with Workbook(file) as wb:
    biolog_data.write_excel(wb, worksheet="all")
    biolog_data.filter(not_in_model_mask).write_excel(wb, worksheet="absent")
    biolog_data.filter(not_in_e_mask).write_excel(wb, worksheet="present")

#### Update the model
---

In [6]:
# Load updates to the model
file = "data/interim/curation/alexis/updates_to_model.xlsx"
metabolites_df = pl.read_excel(file, sheet_name="metabolites")
reactions_df = pl.read_excel(file, sheet_name="reactions")

# Update metabolites
mets_to_add = update_metabolites(metabolites_df)
model.add_metabolites(mets_to_add)

# Update reactions
rxns_to_add = update_reactions(reactions_df)
model.add_reactions(rxns_to_add)

Could not determine dtype for column 5, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 11, falling back to string
Could not determine dtype for column 14, falling back to string
Could not determine dtype for column 15, falling back to string
Could not determine dtype for column 16, falling back to string
Could not determine dtype for column 5, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 12, falling back to string


In [7]:
# Add reaction stoichiometry
for rxn in reactions_df.iter_rows(named=True):
    new_rxn = model.reactions.get_by_id(rxn["rxn_id"])
    new_rxn.build_reaction_from_string(rxn["reaction"])
model

Name,iHlct
Memory address,79c8385fe4a0
Number of metabolites,1782
Number of reactions,2311
Number of genes,769
Number of groups,0
Objective expression,1.0*BIOMASS_hlacus_auto - 1.0*BIOMASS_hlacus_auto_reverse_375bd
Compartments,"mitochondrion, chloroplast, cytoplasm, glyoxysome, extracellular, thylakoid"


In [ ]:
# Save the updated model
cobra.io.save_json_model(model, "models/draft/v0.0.1/hlacustris.json")
cobra.io.write_sbml_model(model, "models/draft/v0.0.1/hlacustris.xml")
cobra.io.validate_sbml_model("models/draft/v0.0.1/hlacustris.xml")
write_excel(model, "models/draft/v0.0.1/hlacustris.xlsx")
model = cobra.io.read_sbml_model("models/draft/v0.0.1/hlacustris.xml")

#### Simulations
---

In [40]:
# Make base medium for plates PM1, PM2 and PM3
# Carbon free
pm12_medium = deepcopy(model.medium)
pm12_medium.pop("EX_co2_e")
# Nitrogen free
pm3_medium = deepcopy(model.medium)
pm3_medium.pop("EX_no3_e")

# Do simulations
num_tol = 1e-6
sim_values = [] # growth rates
sim_bool = [] # Boolean of growth rate
labels = []
for row in biolog_data.iter_rows(named=True):
    if row["e"]:
        # Add metabolite to the medium
        if row["plate"] == "PM1" or row["plate"] == "PM2":
            medium = deepcopy(pm12_medium)
        else:
            medium = deepcopy(pm3_medium)
        met_to_test = "EX_" + row["e"]
        medium[met_to_test] = 1000
        if row["condition"] == "dark":
            medium["EX_photonVis_e"] = 0
        # Do simulations
        with model as m:
            try:
                m.medium = medium
            except KeyError:
                print(f"Exchange reaction not in model for: {met_to_test}")
            sol = m.slim_optimize()

            # Classify the solution as TP, FP, TN or FN
            y_pred = sol>num_tol
            y_true = row["decision"]
            sim_values.append(round(sol,3))
            sim_bool.append(y_pred)
            if y_true == 1 and y_pred == 1:
                labels.append("TP")
            elif y_true == 0 and y_pred == 1:
                labels.append("FP")
            elif y_true == 0 and y_pred == 0:
                labels.append("TN")
            elif y_true == 1 and y_pred == 0:
                labels.append("FN")
    elif row["decision"]:
        y_pred = False
        y_true = row["decision"]
        sim_values.append(0.0)
        sim_bool.append(y_pred)
        if y_true == 1 and y_pred == 1:
            labels.append("TP")
        elif y_true == 0 and y_pred == 1:
            labels.append("FP")
        elif y_true == 0 and y_pred == 0:
            labels.append("TN")
        elif y_true == 1 and y_pred == 0:
            labels.append("FN")
    else:
        y_pred = False
        y_true = row["decision"]
        sim_values.append(0.0)
        sim_bool.append(y_pred)
        if y_true == 1 and y_pred == 1:
            labels.append("TP")
        elif y_true == 0 and y_pred == 1:
            labels.append("FP")
        elif y_true == 0 and y_pred == 0:
            labels.append("TN")
        elif y_true == 1 and y_pred == 0:
            labels.append("FN")

# Append the results to the dataframe
sim_results = {"sim_growth": sim_values, "sim_value": sim_bool, "label": labels}
biolog_data = biolog_data.with_columns(
    [pl.Series(name=key, values=value) for key, value in sim_results.items()]
)

Exchange reaction not in model for: EX_Glc_aD_e
Exchange reaction not in model for: EX_arg__L_e
Exchange reaction not in model for: EX_his__L_e
Exchange reaction not in model for: EX_arg__L_e
Exchange reaction not in model for: EX_his__L_e
Exchange reaction not in model for: EX_Glc_aD_e
Exchange reaction not in model for: EX_arg__L_e
Exchange reaction not in model for: EX_his__L_e
Exchange reaction not in model for: EX_arg__L_e
Exchange reaction not in model for: EX_his__L_e


In [25]:
file = "data/2_processed/biolog_sims.xlsx"
# Save the results
with Workbook(file) as wb:
    biolog_data.write_excel(wb, worksheet="data")

#### Calculate metrics
---

In [42]:
# Get the confusion_matrix and the matthews_corrcoef
has_bigg_mask = [bool(row["bigg"]) for row in biolog_data.iter_rows(named=True)]
simulations = biolog_data.filter(has_bigg_mask)
y_true = list(simulations["decision"])
y_pred = list(simulations["sim_value"])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel().tolist()
corr_coef = matthews_corrcoef(y_true, y_pred)
results = {"TN": tn, "FP": fp, "FN": fn, "TP": tp, "matthews": corr_coef, "tested": len(y_true)}
file = "data/2_processed/confusion_matrix.txt"
with open(file, "w") as f:
    json.dump(results, f, indent=4)
results

{'TN': 337,
 'FP': 34,
 'FN': 42,
 'TP': 1,
 'matthews': -0.07499804468867463,
 'tested': 414}

In [43]:
# Metrics by condition: light
simulations = biolog_data.filter(has_bigg_mask, pl.col("condition")=="light")
y_true = list(simulations["decision"])
y_pred = list(simulations["sim_value"])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel().tolist()
corr_coef = matthews_corrcoef(y_true, y_pred)
results = {"TN": tn, "FP": fp, "FN": fn, "TP": tp, "matthews": corr_coef, "tested": len(y_true)}
results

{'TN': 142,
 'FP': 23,
 'FN': 41,
 'TP': 1,
 'matthews': -0.14518934067457712,
 'tested': 207}

In [44]:
# Metrics by condition: dark
simulations = biolog_data.filter(has_bigg_mask, pl.col("condition")=="dark")
y_true = list(simulations["decision"])
y_pred = list(simulations["sim_value"])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel().tolist()
corr_coef = matthews_corrcoef(y_true, y_pred)
results = {"TN": tn, "FP": fp, "FN": fn, "TP": tp, "matthews": corr_coef, "tested": len(y_true)}
results

{'TN': 195,
 'FP': 11,
 'FN': 1,
 'TP': 0,
 'matthews': -0.016505728481847302,
 'tested': 207}

#### Test the model
---

In [12]:
# Print the solution to the model in standar media
sol = model.optimize()
model.summary(solution=sol)

Metabolite,Reaction,Flux,C-Number,C-Flux
gam_e,EX_gam_e,61.45,6,4.08%
malt_e,EX_malt,722.3,12,95.92%
mg2_e,EX_mg2_e,0.6035,0,0.00%
na1_e,EX_na1_e,836.6,0,0.00%
no3_e,EX_no3_e,229,0,0.00%
o2_e,EX_o2_e,416.6,0,0.00%
photonVis_e,EX_photonVis_e,1000,0,0.00%
pi_e,EX_pi_e,22.92,0,0.00%
so4_e,EX_so4_e,3.238,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux


In [13]:
# Find active reactions for a metabolite
num_tol = 1e-6
met_id = "gua_c"
for rxn in model.reactions:
    if met_id in rxn.build_reaction_string():
        if abs(sol[rxn.id]) >= num_tol:
            print(rxn.id, rxn.build_reaction_string(), sol[rxn.id])

In [20]:
# Find all reactions for a metabolite
met_id = "2obut"
for rxn in model.reactions:
    if met_id in rxn.build_reaction_string():
        print(rxn.id, rxn.build_reaction_string())

SHSL4h h2o_h + suchms_h --> 2obut_h + h_h + nh4_h + succ_h
ACCAHh 1acpc_h + h2o_h <=> 2obut_h + nh4_h
ACCAHm 1acpc_m + h2o_m <=> 2obut_m + nh4_m
PCFPTh 2obut_h + coa_h --> for_h + ppcoa_h
PCFPTm 2obut_m + coa_m --> for_m + ppcoa_m
ACAS_2ahbut 2ahethmpp_h + 2obut_h --> 2ahbut_h + thmpp_h
TAL thr__L_h --> 2obut_h + nh4_h
cyscyslCv cyst__L_h + h2o_h --> 2obut_h + cys__L_h + nh4_h


In [21]:
# Add demand reactions and test the model for a metabolite
met_to_test = "for_e"
demand_to_test = "nh4_h"
with model as m:
    medium = deepcopy(pm12_medium)
    medium["EX_" + met_to_test] = 1000
    m.medium = medium

    dm_met = m.metabolites.get_by_id(demand_to_test)
    m.add_boundary(dm_met, type="demand")
    m.objective = "DM_" + demand_to_test
    sol = m.optimize()
    print(m.summary(solution=sol))

Objective
1.0 DM_nh4_h = 1000.0

Uptake
------
Metabolite  Reaction Flux  C-Number  C-Flux
     h2o_e  EX_h2o_e  500         0   0.00%
       h_e    EX_h_e  500         0   0.00%
    urea_e EX_urea_e  500         1 100.00%

Secretion
---------
Metabolite Reaction  Flux  C-Number  C-Flux
     nh4_h DM_nh4_h -1000         0   0.00%
     co2_e EX_co2_e  -500         1 100.00%



In [27]:
# Add demand reactions and test the model for a growth
met_to_test = "urea_e"
demand_to_test = "nh4_h"
with model as m:
    medium = deepcopy(pm3_medium)
    medium["EX_" + met_to_test] = 1000
    m.medium = medium

    dm_met = m.metabolites.get_by_id(demand_to_test)
    #m.add_boundary(dm_met, type="demand")
    sol = m.optimize()
    print(m.slim_optimize())
    print(m.summary(solution=sol))

0.513068926285173
Objective
1.0 BIOMASS_hlacus_auto = 0.513068926285173

Uptake
------
 Metabolite       Reaction    Flux  C-Number C-Flux
      co2_e       EX_co2_e   158.7         1 99.56%
      h2o_e       EX_h2o_e     127         0  0.00%
        h_e         EX_h_e   1.288         0  0.00%
      mg2_e       EX_mg2_e 0.01366         0  0.00%
       o2_e        EX_o2_e   1.213         0  0.00%
photonVis_e EX_photonVis_e    1000         0  0.00%
       pi_e        EX_pi_e   0.519         0  0.00%
      so4_e       EX_so4_e 0.07331         0  0.00%
     urea_e      EX_urea_e  0.6956         1  0.44%

Secretion
---------
 Metabolite       Reaction   Flux  C-Number C-Flux
      o2D_u       DM_o2D_u   -165         0  0.00%
photon646_h DM_photon646_h -265.7         0  0.00%
photon680_u DM_photon680_u -65.17         0  0.00%



In [17]:
biolog_data.filter(

plate,source,condition,value,peptide,new_bigg,bigg,metacyc,m,h,c,x,e,u,sim_growth,sim_value,label
str,str,str,bool,str,str,str,str,str,str,str,str,str,null,f64,bool,str
"""PM1""","""L-Arabinose""","""light""",true,null,null,"""arab__L""",null,null,null,"""arab__L_c""",null,null,null,null,null,null
"""PM1""","""N-Acetyl-D-Glucosamine""","""light""",true,null,null,"""acgam""",null,null,null,null,null,null,null,null,null,null
"""PM1""","""D-Saccharic Acid""","""light""",true,null,null,"""glcr""",null,null,null,null,null,null,null,null,null,null
"""PM1""","""Succinic Acid""","""light""",true,null,null,"""succ""",null,"""succ_m""","""succ_h""","""succ_c""",null,"""succ_e""",null,22.664,true,"""TP"""
"""PM1""","""D-Galactose""","""light""",true,null,null,"""gal""",null,null,"""gal_h""","""gal_c""",null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""PM3""","""Ethylenediamine""","""dark""",false,null,null,null,"""CPD-3682""",null,null,null,null,null,null,null,null,null
"""PM3""","""L-Alanine""","""dark""",false,null,null,"""ala__L""",null,"""ala__L_m""","""ala__L_h""","""ala__L_c""","""ala__L_x""",null,null,null,null,null
"""PM3""","""D,L-a-Amino-Caprylic Acid""","""dark""",false,null,null,null,null,null,null,null,null,null,null,null,null,null
